# IterativeFib (C++)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/IterativeFib_cpp.ipynb)

Capitolo 14 — Ricorsione. Notebook eseguibile di `IterativeFib`: l'algoritmo in C++, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/IterativeFib_python.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/IterativeFib_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione_cpp.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Ogni programma C++ è scritto su un file dalla cella `%%writefile`, poi compilato con `g++` ed eseguito nella cella successiva: tutto avviene in Colab.

**Nomi.** In questo capitolo non ci sono array: variabili e parametri hanno gli stessi nomi dello pseudocodice. Lo stack e i contatori sono variabili globali, e la traccia si attiva con `trace = true`.

## Fibonacci iterativo: IterativeFib

**Input:** un numero naturale $n$. **Output:** $F(n)$, calcolato da $F(0)$ e $F(1)$ in su tenendo solo gli ultimi due valori. $S(n) = 5n + 1 = \Theta(n)$ passi. Invariante: prima di ogni valutazione della condizione, $prev = F(i-2)$ e $curr = F(i-1)$.

Pseudocodice: sezione 14.5.2 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

In [1]:
%%writefile iterative_fib.cpp
#include <cstdarg>
#include <cstdio>
#include <cmath>
#include <initializer_list>
#include <string>
using namespace std;

long long cnt[20];      // cnt[r] = numero di esecuzioni della riga r (numerazione delle dispense)
long long calls = 0;    // numero di chiamate = nodi dell'albero delle chiamate
int maxDepth = 0;       // profondita' della ricorsione = altezza massima dello stack
long long mults = 0;    // moltiplicazioni (FastPower)
long long moves = 0;    // mosse (Hanoi)
int ev = 0;             // contatore degli eventi della traccia
int stk[1000];          // lo stack delle chiamate: gli argomenti delle chiamate attive (al massimo 1000)
int sp = 0;             // numero di chiamate attive
long long seen[64];     // seen[x] = numero di chiamate con argomento x
bool trace = false;     // true: stampa la traccia dell'esecuzione

void Reset()
{
    for (int r = 0; r < 20; r++) cnt[r] = 0;
    for (int k = 0; k < 64; k++) seen[k] = 0;
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0; sp = 0;
}

long long Total()
{
    long long s = 0;
    for (int r = 0; r < 20; r++) s += cnt[r];
    return s;
}

void PrintSteps(initializer_list<int> lines)
{
    printf("Riga | Esecuzioni\n");
    for (int r : lines) printf("%4d | %10lld\n", r, cnt[r]);
    printf("Passi totali S = %lld\n", Total());
}

string Str(const char* fmt, ...)       // come sprintf, ma restituisce una stringa
{
    char buf[256];
    va_list ap;
    va_start(ap, fmt);
    vsnprintf(buf, sizeof buf, fmt, ap);
    va_end(ap);
    return string(buf);
}

void Push(int x)                       // inizia una chiamata: il suo frame va in cima allo stack
{
    calls++;
    stk[sp++] = x;
    if (sp > maxDepth) maxDepth = sp;
    if (x >= 0 && x < 64) seen[x]++;
}

void Pop()                             // la chiamata termina: il suo frame esce dallo stack
{
    sp--;
}

string StackStr()                      // "[4, 3, 2]"
{
    string s = "[";
    for (int k = 0; k < sp; k++)
    {
        if (k > 0) s += ", ";
        s += to_string(stk[k]);
    }
    return s + "]";
}

void Event(const string& text)         // riga numerata della traccia, con lo stack
{
    ev++;
    printf("%2d. %s   stack: %s\n", ev, text.c_str(), StackStr().c_str());
}

void Say(const string& call, const string& action)   // riga della traccia: la chiamata, rientrata di un livello per ogni chiamata attiva
{
    printf("%-28s| %s\n", (string(2 * (sp - 1), ' ') + call).c_str(), action.c_str());
}

// IterativeFib(n)
long long IterativeFib(int n, bool trace = false)
{
    cnt[2]++;
    if (n <= 1)
    {
        cnt[3]++;
        return n;
    }
    long long prev = 0; cnt[5]++;
    long long curr = 1; cnt[6]++;
    int i = 2; cnt[7]++;
    if (trace)                                     // traccia: riga di inizializzazione
        printf("%-10s | %-8s | %-18s | %4lld | %4lld\n", "inizio", "2", "-", prev, curr);
    cnt[8]++;
    while (i <= n)
    {
        long long next = prev + curr; cnt[9]++;
        prev = curr; cnt[10]++;
        curr = next; cnt[11]++;
        i = i + 1; cnt[12]++;
        cnt[8]++;
        if (trace)                                 // traccia: i - 2 e' il numero dell'iterazione
            printf("%-10d | %-8s | %-18s | %4lld | %4lld\n", i - 2, Str("%d -> %d", i - 1, i).c_str(), Str("%lld + %lld = %lld", next - prev, prev, next).c_str(), prev, curr);
    }
    cnt[14]++;
    return curr;
}

#ifndef SOLO_LIBRERIA
int main()
{
    // Esempio delle dispense: IterativeFib(4)
    int n = 4;
    Reset();
    printf("iterazione | i        | next = prev + curr | prev | curr\n");
    long long r = IterativeFib(n, true);
    printf("Il ciclo termina con i = %d > %d: restituisce curr = %lld\n", n + 1, n, r);
    PrintSteps({2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14});
    printf("S(n) = 5n + 1 = %d\n", 5 * n + 1);

    // Confronto con Fib: passi di Fib = 4F(n+1) - 2 (formula verificata nella sezione precedente)
    printf("\n");
    printf("  n |   F(n) | Fib: 4F(n+1) - 2 | IterativeFib | 5n + 1\n");
    int values[] = {5, 10, 20, 30};
    for (int v : values)
    {
        Reset();
        r = IterativeFib(v);
        long long steps = Total();
        printf("%3d | %6lld | %16lld | %12lld | %d\n", v, r, 4 * IterativeFib(v + 1) - 2, steps, 5 * v + 1);
    }
    return 0;
}
#endif

Writing iterative_fib.cpp


In [2]:
!g++ -std=c++17 -O2 -o iterative_fib iterative_fib.cpp && ./iterative_fib

iterazione | i        | next = prev + curr | prev | curr
inizio     | 2        | -                  |    0 |    1
1          | 2 -> 3   | 0 + 1 = 1          |    1 |    1
2          | 3 -> 4   | 1 + 1 = 2          |    1 |    2
3          | 4 -> 5   | 1 + 2 = 3          |    2 |    3
Il ciclo termina con i = 5 > 4: restituisce curr = 3
Riga | Esecuzioni
   2 |          1
   3 |          0
   5 |          1
   6 |          1
   7 |          1
   8 |          4
   9 |          3
  10 |          3
  11 |          3
  12 |          3
  14 |          1
Passi totali S = 21
S(n) = 5n + 1 = 21

  n |   F(n) | Fib: 4F(n+1) - 2 | IterativeFib | 5n + 1
  5 |      5 |               30 |           26 | 26
 10 |     55 |              354 |           51 | 51
 20 |   6765 |            43782 |          101 | 101
 30 | 832040 |          5385074 |          151 | 151


**Prova tu.** Cambia $n$ e controlla che i passi siano $5n + 1$.

In [3]:
%%writefile prova_iterative_fib.cpp
#define SOLO_LIBRERIA   // non eseguire il main di iterative_fib.cpp
#include "iterative_fib.cpp"

int main()
{
    int n = 10;        // <-- cambia n
    Reset();
    printf("iterazione | i        | next = prev + curr | prev | curr\n");
    long long r = IterativeFib(n, true);
    printf("F(%d) = %lld, S = %lld = 5n + 1 = %d\n", n, r, Total(), 5 * n + 1);
    return 0;
}

Writing prova_iterative_fib.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_iterative_fib prova_iterative_fib.cpp && ./prova_iterative_fib

iterazione | i        | next = prev + curr | prev | curr
inizio     | 2        | -                  |    0 |    1
1          | 2 -> 3   | 0 + 1 = 1          |    1 |    1
2          | 3 -> 4   | 1 + 1 = 2          |    1 |    2
3          | 4 -> 5   | 1 + 2 = 3          |    2 |    3
4          | 5 -> 6   | 2 + 3 = 5          |    3 |    5
5          | 6 -> 7   | 3 + 5 = 8          |    5 |    8
6          | 7 -> 8   | 5 + 8 = 13         |    8 |   13
7          | 8 -> 9   | 8 + 13 = 21        |   13 |   21
8          | 9 -> 10  | 13 + 21 = 34       |   21 |   34
9          | 10 -> 11 | 21 + 34 = 55       |   34 |   55
F(10) = 55, S = 51 = 5n + 1 = 51


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).